In [6]:
import numpy as np
import os 
from playsound import playsound

from mic import Microphone
from silero_vad import Silero_VAD
from pho_asr import PhoASR
from ICA import ICASeparator
import torch 


playsound is relying on another python subprocess. Please use `pip install pygobject` if you want playsound to run more efficiently.


In [7]:
audio = Microphone()
asr_module = PhoASR(model_name= 'vinai/PhoWhisper-large')

ALSA lib pcm_dsnoop.c:567:(snd_pcm_dsnoop_open) unable to open slave
ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) unable to open slave
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) unable to open slave
Cannot connect to server socket err = No such file or directory
Cannot connect to server request channel
jack server is not running or cannot be started
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock


--- Initializing PhoASR with model: 'vinai/PhoWhisper-large' ---
✅ Model loaded successfully on device: 'cuda'.


In [8]:
audio_path = 'recording_at_2025-10-21_21-52-22.wav'
playsound(audio_path)

PlaysoundException: Cannot find a sound with filename: recording_at_2025-10-21_21-52-22.wav

#### Using ICA

In [ ]:
mixed_audio_path = 'audio_sources/combined_output.wav'
separator = ICASeparator(n_components=2)
separator.process(mixed_audio_path)

=> Dont think ICA is enough for separate the signal 

#### Conv-Tasnet

In [ ]:
from asteroid.models import ConvTasNet
import torchaudio 

In [ ]:
from asteroid.models import ConvTasNet
import torchaudio
import torch

def separate_speakers_from_mono(input_path, max_speakers=3, output_prefix="speaker_"):
    """
    Separate multiple speakers from mono microphone recording.
    
    Args:
        input_path (str): Path to mono audio recording
        max_speakers (int): Maximum number of speakers to separate
        output_prefix (str): Prefix for output files
    """
    # Load pre-trained model (trained for speaker separation)
    model = ConvTasNet.from_pretrained("JorisCos/ConvTasNet_Libri2Mix_sepnoisy_16k")
    
    # Load mono audio
    waveform, sample_rate = torchaudio.load(input_path)
    
    # Ensure proper shape for model
    if len(waveform.shape) == 1:
        waveform = waveform.unsqueeze(0)
    
    # Perform separation
    separated_sources = model.separate(waveform)
    
    # Filter out low-energy sources (likely noise or silence)
    valid_sources = []
    energy_threshold = 0.001  # Adjust based on your audio
    
    for i, source in enumerate(separated_sources):
        energy = torch.mean(source.abs()).item()
        if energy > energy_threshold:
            valid_sources.append((i, source, energy))
            print(f"Speaker {i+1}: Energy = {energy:.4f}")
    
    # Sort by energy (strongest speakers first)
    valid_sources.sort(key=lambda x: x[2], reverse=True)
    
    # # Save separated speakers
    # for idx, (original_idx, source, energy) in enumerate(valid_sources[:max_speakers]):
    #     output_path = f"{output_prefix}{idx+1}.wav"
    #     torchaudio.save(output_path, source.unsqueeze(0), sample_rate)
    #     print(f"Saved speaker {idx+1} to {output_path}")
    
    return len(valid_sources)

# Usage
audio_path = "audio_sources/mix_nhahang.wav"
num_speakers = separate_speakers_from_mono(audio_path)
print(f"Detected {num_speakers} speakers")

/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.S

Speaker 1: Energy = 0.0731
Detected 1 speakers


In [ ]:
# Install required packages first: pip install speechbrain torch torchaudio

import torch
import torchaudio
from speechbrain.pretrained import SepformerSeparation as separator
import os

def separate_speakers_speechbrain(input_path, output_dir="separated_audio"):
    """
    Separate speakers using SpeechBrain's Sepformer model.
    
    Args:
        input_path (str): Path to mixed audio file
        output_dir (str): Directory to save separated audio files
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load pre-trained Sepformer model
    model = separator.from_hparams(
    source="speechbrain/sepformer-wsj02mix",
    savedir='pretrained_models/sepformer-wsj02mix'
)
    # Load and preprocess audio
    waveform, sample_rate = torchaudio.load(input_path)
    
    # Ensure mono input
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Perform separation
    est_sources = model.separate_batch(waveform)
    
    # Save separated sources
    separated_files = []
    for i, source in enumerate(est_sources.squeeze(0)):
        output_path = os.path.join(output_dir, f"speaker_{i+1}.wav")
        torchaudio.save(output_path, source.unsqueeze(0), sample_rate)
        separated_files.append(output_path)
        print(f"Saved speaker {i+1} to {output_path}")
    
    return separated_files

# Alternative: Using WHAM! dataset trained model for better performance
def separate_speakers_wham(input_path, output_dir="separated_wham"):
    """
    Use WHAM! trained model for better noise robustness.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Load WHAM! trained model (better for noisy environments)
    model = separator.from_hparams(
        source="speechbrain/sepformer-wham16k-enhancement", 
        savedir='pretrained_models/sepformer-wham16k-enhancement'
    )
    
    waveform, sample_rate = torchaudio.load(input_path)
    
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Separate sources
    est_sources = model.separate_batch(waveform.unsqueeze(0))
    
    separated_files = []
    for i, source in enumerate(est_sources.squeeze(0)):
        output_path = os.path.join(output_dir, f"enhanced_speaker_{i+1}.wav")
        torchaudio.save(output_path, source.unsqueeze(0), sample_rate)
        separated_files.append(output_path)
        print(f"Enhanced speaker {i+1} saved to {output_path}")
    
    return separated_files

# Usage examples
audio_path = "audio_sources/mix_nhahang.wav"

# Method 1: Standard separation
separated_files = separate_speakers_speechbrain(audio_path)

# Method 2: Enhanced separation (better for noisy environments like restaurants)
enhanced_files = separate_speakers_wham(audio_path)

/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3

Saved speaker 1 to separated_audio/speaker_1.wav
Saved speaker 2 to separated_audio/speaker_2.wav
Saved speaker 3 to separated_audio/speaker_3.wav
Saved speaker 4 to separated_audio/speaker_4.wav
Saved speaker 5 to separated_audio/speaker_5.wav
Saved speaker 6 to separated_audio/speaker_6.wav
Saved speaker 7 to separated_audio/speaker_7.wav
Saved speaker 8 to separated_audio/speaker_8.wav
Saved speaker 9 to separated_audio/speaker_9.wav
Saved speaker 10 to separated_audio/speaker_10.wav
Saved speaker 11 to separated_audio/speaker_11.wav
Saved speaker 12 to separated_audio/speaker_12.wav
Saved speaker 13 to separated_audio/speaker_13.wav
Saved speaker 14 to separated_audio/speaker_14.wav
Saved speaker 15 to separated_audio/speaker_15.wav
Saved speaker 16 to separated_audio/speaker_16.wav
Saved speaker 17 to separated_audio/speaker_17.wav
Saved speaker 18 to separated_audio/speaker_18.wav
Saved speaker 19 to separated_audio/speaker_19.wav
Saved speaker 20 to separated_audio/speaker_20.wa

KeyboardInterrupt: 

In [9]:
from speechbrain.inference.separation import SepformerSeparation as separator
import torchaudio

# Load the pre-trained model from Hugging Face
model = separator.from_hparams(
    source="speechbrain/sepformer-wsj02mix",
    savedir='pretrained_models/sepformer-wsj02mix'
)

# Separate an audio file (expects 8kHz, single-channel)
# The model will automatically download the file from the path if it's a URL
est_sources = model.separate_file(path='audio_sources/mix_nhahang.wav')

# Save the separated sources
torchaudio.save("source1_estimate.wav", est_sources[:, :, 0].detach().cpu(), 8000)
torchaudio.save("source2_estimate.wav", est_sources[:, :, 1].detach().cpu(), 8000)

/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https:/

Resampling the audio from 44100 Hz to 8000 Hz


/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/home/lequocthinh/miniconda3/envs/deeplearning/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3